# 01. Data preprocessing

This notebook prepares task-specific train/test files from the baseline ADNI clinical dataframe.

The main steps are:

- load the baseline clinical dataframe
- define the binary classification tasks
- create a shared stratified train/test split
- construct task-specific binary labels
- encode categorical variables using training-derived mappings
- convert numerical variables and handle missing values
- save processed train/test files for each task
- save task-wise sample count summary

## 1. Import libraries

This section imports the basic libraries used for data handling, random seed control, and train/test splitting.

In [9]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

## 2. Reproducibility setting

A fixed random seed is used to make the train/test split reproducible.

In [10]:
GLOBAL_SEED = 17

def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

set_seed(GLOBAL_SEED)

## 3. File paths and protocol settings

The raw ADNI clinical dataframe is not included in this repository due to data use agreements.  
Users should place their local ADNI clinical dataframe under `data/raw/` or modify the path below.

In [11]:
RAW_CSV = "../data/raw/adni_clinical_dataframe.csv"

PROCESSED_DIR = "../data/processed"
SPLIT_DIR = "../data/splits"

OUTER_TEST_SIZE = 0.10
STRATIFY_COL = "DIAGNOSIS2"

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLIT_DIR, exist_ok=True)

## 4. Task definition

Four binary classification tasks are defined:

- AD vs. CN
- AD vs. MCI
- MCI vs. CN
- pMCI vs. sMCI

For each task, the positive class is encoded as 1 and the negative class is encoded as 0.

In [12]:
DROP_COLS_CANDIDATES = [
    "PHASE", "PTID", "RID", "VISCODE2",
    "DIAGNOSIS", "DIAGNOSIS2",
    "CDGLOBAL"
]

TASKS = [
    {
        "name": "AD_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["CN"],
    },
    {
        "name": "AD_vs_MCI",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["AD"],
        "neg_labels": ["MCI"],
    },
    {
        "name": "MCI_vs_CN",
        "label_col": "DIAGNOSIS",
        "pos_labels": ["MCI"],
        "neg_labels": ["CN"],
    },
    {
        "name": "pMCI_vs_sMCI",
        "label_col": "DIAGNOSIS2",
        "pos_labels": ["pMCI"],
        "neg_labels": ["sMCI"],
    },
]

## 5. Label mapping functions

The following functions select task-specific samples and convert diagnostic labels into binary labels.

In [13]:
def assert_binary(y, where=""):
    classes = np.unique(y)

    if len(classes) < 2:
        raise ValueError(f"{where}: only one class present: {classes}")

def map_labels(df: pd.DataFrame, label_col: str, pos_labels, neg_labels):
    labels = df[label_col].astype(str).str.strip()

    pos_set = {str(x).strip() for x in pos_labels}
    neg_set = {str(x).strip() for x in neg_labels}

    mask = labels.isin(pos_set.union(neg_set))
    sub_df = df.loc[mask].copy().reset_index(drop=True)

    y = sub_df[label_col].astype(str).str.strip().isin(pos_set).astype(int).values

    return sub_df, y

## 6. Preprocessing utility functions

Categorical variables are encoded using mappings learned from the training subset only.  
Unseen categories in the test set are mapped to 0.

Numerical variables are converted to numeric values, and missing or non-parsable values are replaced with 0 at this stage.  
Z-score standardization is not performed here. It is performed during model training using the inner-training subset only.

In [14]:
def is_cat_col(series: pd.Series) -> bool:
    dtype = series.dtype

    if pd.api.types.is_object_dtype(dtype) or pd.api.types.is_bool_dtype(dtype):
        return True

    return isinstance(dtype, pd.CategoricalDtype)

def build_preprocessor(train_df: pd.DataFrame, feature_cols):
    cat_cols, num_cols = [], []

    for col in feature_cols:
        if is_cat_col(train_df[col]):
            cat_cols.append(col)
        else:
            num_cols.append(col)

    cat_maps = {}

    for col in cat_cols:
        values = train_df[col].astype("object")
        values = values.where(~values.isna(), "__MISSING__").astype(str)
        unique_values = pd.unique(values)

        # 0 is reserved for unseen categories in validation/test data.
        cat_maps[col] = {val: i + 1 for i, val in enumerate(unique_values)}

    return cat_cols, num_cols, cat_maps

def transform_df(df: pd.DataFrame, feature_cols, cat_cols, num_cols, cat_maps):
    X = np.zeros((len(df), len(feature_cols)), dtype=np.float32)

    for j, col in enumerate(feature_cols):
        if col in cat_cols:
            values = df[col].astype("object")
            values = values.where(~values.isna(), "__MISSING__").astype(str)

            X[:, j] = np.array(
                [cat_maps[col].get(v, 0) for v in values],
                dtype=np.float32
            )

        else:
            values = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
            X[:, j] = np.nan_to_num(values.values, nan=0.0)

    return X

## 7. Load ADNI clinical dataframe

The input dataframe should contain participant identifiers, diagnostic labels, and baseline clinical variables.

In [ ]:
df_all = pd.read_csv(RAW_CSV).copy()

df_all[STRATIFY_COL] = df_all[STRATIFY_COL].astype(str).str.strip()
df_all = df_all[df_all[STRATIFY_COL].notna()].copy()
df_all = df_all[df_all[STRATIFY_COL] != ""].copy()

print("Data shape:", df_all.shape)
df_all.head()

## 8. Check diagnostic distribution

Before splitting the dataset, the distribution of the stratification label is checked.

In [ ]:
print("Diagnosis distribution:")
print(df_all[STRATIFY_COL].value_counts())

## 9. Shared stratified train/test split

A single shared outer train/test split is created and reused across all binary classification tasks.  
This makes the evaluation protocol consistent across tasks.

In [ ]:
df_train_all, df_test_all = train_test_split(
    df_all,
    test_size=OUTER_TEST_SIZE,
    random_state=GLOBAL_SEED,
    stratify=df_all[STRATIFY_COL].values,
)

df_train_all = df_train_all.reset_index(drop=True)
df_test_all = df_test_all.reset_index(drop=True)

print("Train samples:", len(df_train_all))
print("Test samples:", len(df_test_all))

print("\nTrain distribution:")
print(df_train_all[STRATIFY_COL].value_counts())

print("\nTest distribution:")
print(df_test_all[STRATIFY_COL].value_counts())

## 10. Generate task-specific train/test files

For each binary task, task-specific samples are selected from the shared outer split.  
Categorical mappings are fitted only on the task-specific training subset and then applied to the corresponding test subset.

In [ ]:
task_summaries = []

for task in TASKS:
    task_name = task["name"]

    # Select task-specific samples from the shared outer split.
    df_train_task, y_train = map_labels(
        df_train_all,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    df_test_task, y_test = map_labels(
        df_test_all,
        task["label_col"],
        task["pos_labels"],
        task["neg_labels"],
    )

    assert_binary(y_train, f"{task_name} train")
    assert_binary(y_test, f"{task_name} test")

    # Select feature columns.
    drop_cols = set(DROP_COLS_CANDIDATES + [task["label_col"]])
    feature_cols = [col for col in df_train_task.columns if col not in drop_cols]

    # Fit categorical mappings on training data only.
    cat_cols, num_cols, cat_maps = build_preprocessor(df_train_task, feature_cols)

    # Transform train and test using training-derived mappings.
    X_train = transform_df(df_train_task, feature_cols, cat_cols, num_cols, cat_maps)
    X_test = transform_df(df_test_task, feature_cols, cat_cols, num_cols, cat_maps)

    # Save processed train/test files.
    train_out = pd.DataFrame(X_train, columns=feature_cols)
    train_out.insert(0, "label", y_train)

    test_out = pd.DataFrame(X_test, columns=feature_cols)
    test_out.insert(0, "label", y_test)

    train_path = f"{PROCESSED_DIR}/{task_name}_train.csv"
    test_path = f"{PROCESSED_DIR}/{task_name}_test.csv"

    train_out.to_csv(train_path, index=False)
    test_out.to_csv(test_path, index=False)

    # Save feature list.
    feature_path = f"{SPLIT_DIR}/{task_name}_features.txt"

    with open(feature_path, "w", encoding="utf-8") as f:
        for col in feature_cols:
            f.write(col + "\n")

    task_summaries.append({
        "task": task_name,
        "positive_class": task["pos_labels"][0],
        "negative_class": task["neg_labels"][0],
        "n_features": len(feature_cols),
        "train_n": len(y_train),
        "test_n": len(y_test),
        "positive_train_n": int((y_train == 1).sum()),
        "negative_train_n": int((y_train == 0).sum()),
        "positive_test_n": int((y_test == 1).sum()),
        "negative_test_n": int((y_test == 0).sum()),
        "train_file": train_path,
        "test_file": test_path,
        "feature_file": feature_path,
    })

task_summary_df = pd.DataFrame(task_summaries)
task_summary_df

## 11. Save task split summary

The task split summary is saved for reproducibility reporting and can be used to prepare the Supplementary Material.

In [ ]:
summary_path = f"{SPLIT_DIR}/task_split_summary.csv"
task_summary_df.to_csv(summary_path, index=False)

print(f"Saved task split summary to: {summary_path}")

## 12. Expected outputs

After running this notebook, the following files should be generated locally:

```text
data/processed/AD_vs_CN_train.csv
data/processed/AD_vs_CN_test.csv
data/processed/AD_vs_MCI_train.csv
data/processed/AD_vs_MCI_test.csv
data/processed/MCI_vs_CN_train.csv
data/processed/MCI_vs_CN_test.csv
data/processed/pMCI_vs_sMCI_train.csv
data/processed/pMCI_vs_sMCI_test.csv

data/splits/AD_vs_CN_features.txt
data/splits/AD_vs_MCI_features.txt
data/splits/MCI_vs_CN_features.txt
data/splits/pMCI_vs_sMCI_features.txt
data/splits/task_split_summary.csv